### Questions
1. FireplaceQu - 40% of records have nulls - 690 - should we ignore this feature or fill with mode?
2. BsmtExposure - Good, Avg, Min, No exposure, No basement(NULLs) - treat it as Ordinal or Nominal for encoding?
3. BsmtFinType1 & BsmtFinType2 - Good, Avg, Below Avg, Low Quality, No basement, Unfinished
4. Utilities - Since there is only one value in 99.99% cases, do we need to use this feature?
5. NULL / NA Handling - should we fill with mode or should we fill with No (When NA/Nulls mean no basement) and how does that handle in future when we get nulls. should we convert the input test data while sending to the algorithm.

### Overview of the dataset
1. There are 1460 records & 81 columns/features and 1 output column.
2. Output is numeric - So we need to use regression algorithm.
3. Out of 80 features there are 43 non numerical and 37 are numerical.
4. Since there are non numerical features, we have to perform encoding (OneHotEncoding or OrdinalEncoding) before training the model.
6. Since there are numerical features, we might have to performm scaling.
7. We can ignore(drop) Id column as it is just an unique identifier.
8. There are 19 features with nulls. There are 5 features ('Alley', 'MasVnrType ', 'PoolQC', 'Fence', 'MiscFeature') with more than 50% of the data as nulls, we can drop such features from training the model.
9. There are features with less NULL values(<50%), we might have to impute with mean/mode depending on the datatype.
10. There are features with outliers, we have to identify and drop such records.
11. We have to perform feature construction inorder to reduce the features to train the model. There are cetain columns that deal with area for each room, instead we can calculate the entire area and drop individual features.
12. We have to perform CoRelation analysis and identify the features that are highly CoRelated and drop such features.

## Confirmed Code

In [ ]:
# --------------------------------------------------------------
## Handling NULL values
- Based on the above analysis - Drop features that have more than 50% of the values as NULL, except 
- In other features, where <50% nulls are present, Impute NULL values with mean/mode, etc. depending on the data type
# --------------------------------------------------------------

# 50% values are NULLs in these features - 'PoolQC', 'MiscFeature', 'Alley', 'Fence'
# Drop these features.

TrainDF_HousingData.drop(['PoolQC', 'MiscFeature', 'Alley', 'Fence'],axis=1,inplace = True)

# --------------------------------------------------------------
# Impute with Mean
# --------------------------------------------------------------



# --------------------------------------------------------------
# Impute with Mode
# --------------------------------------------------------------

TrainDF_HousingData['BsmtExposure'] = TrainDF_HousingData['BsmtExposure'].fillna(TrainDF_HousingData['BsmtExposure'].mode()[0])
# Once try with No

TrainDF_HousingData['GarageCond'] = TrainDF_HousingData['GarageCond'].fillna(TrainDF_HousingData['GarageCond'].mode()[0])

TrainDF_HousingData['BsmtFinType2'] = TrainDF_HousingData['BsmtFinType2'].fillna(TrainDF_HousingData['BsmtFinType2'].mode()[0])

TrainDF_HousingData['BsmtCond'] = TrainDF_HousingData['BsmtCond'].fillna(TrainDF_HousingData['BsmtCond'].mode()[0])

TrainDF_HousingData['MasVnrArea'] = TrainDF_HousingData['MasVnrArea'].fillna(TrainDF_HousingData['MasVnrArea'].mode()[0])

TrainDF_HousingData['Electrical'] = TrainDF_HousingData['Electrical'].fillna(TrainDF_HousingData['Electrical'].mode()[0])


# --------------------------------------------------------------
# Impute with 'No' instead of Mode - Later on use Mode and see how it is impacting the score.
# --------------------------------------------------------------

TrainDF_HousingData['MasVnrType'] = TrainDF_HousingData['MasVnrType'].fillna('No')
# Even though MasVnrType has 50% values as nulls, let us utilize this feature as not null haves are having more impact on the SalePrice

TrainDF_HousingData['FireplaceQu'] = TrainDF_HousingData['FireplaceQu'].fillna('No')
# FireplaceQu - 40% of values are nulls???

TrainDF_HousingData['GarageType'] = TrainDF_HousingData['GarageType'].fillna('No')
# Try with Mode

TrainDF_HousingData['GarageFinish'] = TrainDF_HousingData['GarageFinish'].fillna('No')
# Try with Mode

TrainDF_HousingData['GarageQual'] = TrainDF_HousingData['GarageQual'].fillna('No')
# Try with mode

TrainDF_HousingData['BsmtFinType1'] = TrainDF_HousingData['BsmtFinType1'].fillna('No')
# Once try with Mode, but mostly keep No as it seems to be correct.

TrainDF_HousingData['BsmtQual'] = TrainDF_HousingData['BsmtQual'].fillna('No')
# No need to check with Mode, No is correct.

# --------------------------------------------------------------
# Impute with 0 instead of Mean - Later use Mean and see how the score changes
# --------------------------------------------------------------

TrainDF_HousingData['GarageYrBlt'] = TrainDF_HousingData['GarageYrBlt'].fillna(0)
# May be we should drop this feature as this has more corelation with YearBuilt
# Do this after completing CoRelatin Analysis of all the columns.

TrainDF_HousingData['LotFrontage'] = TrainDF_HousingData['LotFrontage'].fillna(0)
# Try with mean


# Outliers - Drop
Outlier_Values = [49,  186,  198,  250,  298,  314,  336, 379, 441,  496,  524,  598, 636,  636,  692, 707,  739,  810,  935,  955, 1062, 1183, 1299]
TrainDF_HousingData = TrainDF_HousingData[TrainDF_HousingData.Id.isin(Outlier_Values) == False]


# Feature Construction 
# Create new features from the existing features - self explanatory.

TrainDF_HousingData['House_Age'] = TrainDF_HousingData['YrSold'] - TrainDF_HousingData['YearBuilt']

TrainDF_HousingData['House_Remodel_Age'] = TrainDF_HousingData['YrSold'] - TrainDF_HousingData['YearRemodAdd']

TrainDF_HousingData['Total_Sqft'] = TrainDF_HousingData['1stFlrSF'] + TrainDF_HousingData['2ndFlrSF'] \
+ TrainDF_HousingData['BsmtFinSF1'] + TrainDF_HousingData['BsmtFinSF2']

TrainDF_HousingData['Total_Area'] = TrainDF_HousingData['GrLivArea'] + TrainDF_HousingData['TotalBsmtSF']

TrainDF_HousingData['Total_Baths'] = TrainDF_HousingData['BsmtFullBath'] + TrainDF_HousingData['FullBath'] \
+ 0.5 * (TrainDF_HousingData['BsmtHalfBath'] + TrainDF_HousingData['HalfBath'])

TrainDF_HousingData['Total_Porch_Sqft'] = TrainDF_HousingData['OpenPorchSF'] + TrainDF_HousingData['3SsnPorch'] \
+ TrainDF_HousingData['EnclosedPorch'] + TrainDF_HousingData['ScreenPorch'] + TrainDF_HousingData['WoodDeckSF']

# Remove the features used in Feature Construction
# Remove the features used in Feature Construction - 18 original features are removed

TrainDF_HousingData = TrainDF_HousingData.drop(columns = ['YrSold','YearBuilt','YearRemodAdd','1stFlrSF','2ndFlrSF','BsmtFinSF1','BsmtFinSF2',\
                                                          'GrLivArea','TotalBsmtSF', 'BsmtFullBath','FullBath','BsmtHalfBath','HalfBath',\
                                                          'OpenPorchSF', '3SsnPorch','EnclosedPorch','ScreenPorch','WoodDeckSF'])



# Drop - ID - unique identifier, no significance - drop this feature
TrainDF_HousingData.drop(['Id'],axis=1,inplace = True)

# CoRelation handling
TrainDF_HousingData.drop(['GarageYrBlt'],axis=1,inplace = True)
TrainDF_HousingData.drop(['GarageArea'],axis=1,inplace = True)

# Encoding and Scaling
Numerical_Features = TrainDF_HousingData.select_dtypes(include=['int64','float64']).columns
Numerical_Features = Numerical_Features.drop(['SalePrice'])

# Make a list of only those features that require Ordinal Encoding
Ordinal_Encoding_Features = ['LotShape', 'LandContour','Utilities','LandSlope',  'BsmtQual',  'BsmtFinType1', 'BsmtFinType2', 'CentralAir',  \
                             'Functional', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'PavedDrive', 'ExterCond', 'KitchenQual', 'BsmtExposure', \
                             'HeatingQC','ExterQual', 'BsmtCond','GarageCond']

# Make a list of only those features that require OneHot Encoding
OneHot_Encoding_Features = ['Street', 'LotConfig','Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st', \
                  'Exterior2nd', 'MasVnrType','Foundation',  'Electrical',  'SaleType', 'MSZoning', 'SaleCondition', 'Heating', 'GarageType', \
                  'RoofMatl']



### Usefull code

In [3]:
Confirmed handling:-
---------------------

# Skewed data
TrainDF_HousingData['SalePrice'] = np.log1p(TrainDF_HousingData['SalePrice'])


Numerical_Features_PipelineObj = Pipeline(steps = [
    ('Imputer_NumMean',SimpleImputer(strategy='mean')),
    ('StandardScaler',StandardScaler()) # X_scaled = (X - mean) / standard_deviation.
])

Ordinal_Encoding_Features_PipelineObj = Pipeline(steps = [
    ('Imputer_CatMode',SimpleImputer(strategy='most_frequent')),
    ('OrdinalEncoder',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))
])

OneHot_Encoding_Features_PipelineObj = Pipeline(steps = [
    ('CategoricalModeImputer',SimpleImputer(strategy='most_frequent')),
    ('OHE',OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

ColumnTransformer_Obj = ColumnTransformer(transformers=[
    ('NumTransPip',Numerical_Features_PipelineObj,Numerical_Features),
    ('OrdTransPip',Ordinal_Encoding_Features_PipelineObj,Ordinal_Encoding_Features),
    ('OHETransPip',OneHot_Encoding_Features_PipelineObj,OneHot_Encoding_Features)
], remainder='passthrough',n_jobs=-1)

Pipeline_AllCombined = Pipeline(steps = [
    ('FinalPreprocessingPipeline',ColumnTransformer_Obj)
])

# Prepare the data
X = TrainDF_HousingData.drop('SalePrice',axis=1)
y = TrainDF_HousingData['SalePrice']

X_preprocessed = Pipeline_AllCombined.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_preprocessed,y,test_size=0.2,random_state=25)

# Test various ML Algorithma and get the score

SyntaxError: invalid syntax (3378899483.py, line 6)

In [ ]:
Draft handling:-
-----------------
PCA - Feature Selection and Feature Extraction

# CoRelation Analysis
TestDF_HousingData['GarageYrBlt'].corr(TestDF_HousingData['YearBuilt'])
# 0.83
TrainDF_HousingData.drop(['GarageYrBlt'],axis=1,inplace = True)

# From above CoRelation Matrix, get the Features that are highly impacting the SalePrice, do not drop, these features are most 
# important in training the data

# OverallQual(0.8)
# TotalArea(0.82)
# TotalSqft (0.74)

# From above CoRelation Matrix, we can take Highly CoRelated Features - >= 80% and drop those features.

# GarageCars & GarageArea(0.89) >>> We can drop one of these Feature
# GarageCars - SalePrice (0.66)
# GarageArea - SalePrice(0.65)
TrainDF_HousingData.drop(['GarageArea'],axis=1,inplace = True)
TestDF_HousingData.drop(['GarageArea'],axis=1,inplace = True)

# TotalArea = TotalSqft (0.81)


'''
# Details of the outliers
OverallCond - ID = 379, 692, 1183
YearBuilt ID = 186
BsmtFinSF1 - ID = 1299
BsmtFinSF2 - ID = 441
LotFrontage - ID = 935 and 1299
TotalBsmtSF - ID = 1299
BsmtFullBath - ID - 739
BsmtHalfBath - Id= 598, 955
LotArea - ID = 250, 314, 336, 707. Also check if we need to drop outliers with manual calculation - Housing_Data_All_Master.query('LotArea > 40460.62287932922')
GarageYrBlt - NA
GarageArea ID = 1062
MasVnrArea - ID = 298, and there seems to be 32 records with manual Housing_Data_All_Master.query('MasVnrArea > 646.8838814696435')
OverallQual - Id = 524(may be), 1299 - TrainDF_HousingData.query('OverallQual == 10 and SalePrice < 200000')
LowQualFinSF - Id=186 - TrainDF_HousingData.query('LowQualFinSF > 500 and SalePrice > 400000')
GrLivArea - id = 524, 1299 - TrainDF_HousingData.query('GrLivArea > 4000 and SalePrice < 300000')
BedroomAbvGr - ID = 636 - TrainDF_HousingData.query('BedroomAbvGr == 8')
KitchenAbvGr - ID = 955, 49, 810 - TrainDF_HousingData.query('KitchenAbvGr == 3/2')
TotRmsAbvGrd - ID = 636 - TrainDF_HousingData.query('TotRmsAbvGrd == 14')
OpenPorchSF - ID = 496 - TrainDF_HousingData.query('OpenPorchSF > 500 and SalePrice < 100000')
EnclosedPorch - ID = 198, TrainDF_HousingData.query('EnclosedPorch > 500')
MiscVal
'''

# Scaling
# MSSubClass, LotFrontage, LotArea, GarageYrBlt, MasVnrArea, BsmtFinSF1, BsmtFinSF2, BsmtUnfSF, BsmtFullBath, GarageCars, GarageArea, 
# YearBuilt, YearRemodAdd, 1stFlrSF, 2ndFlrSF, OverallCond, YearBuilt
# OverallQual???

### All feature - analysis pending

### All feature - analysis completed